In [1]:
# ── [Cell 1] Imports & config ─────────────────────────────────────────────────

import struct, os, sys, time, mmap
import numpy as np
from pathlib import Path

STL_PATH      = Path("results_frangi/vessels_binary.stl")
MAX_FACES     = 2_000_000    # faces sent to renderer; reduce if browser lags
SURFACE_COLOR = 0xE05050     # red-ish
BACKGROUND    = 0x0a0a0a
OPACITY       = 0.85


In [2]:
# ── [Cell 2] CUDA kernel ──────────────────────────────────────────────────────
%env CUDA_VISIBLE_DEVICES=4   # set cuda device (optional)
_CUDA_READ_FACES = r"""
/*
 * read_binary_stl_faces
 * ─────────────────────
 * One thread per face. Binary STL record (50 bytes):
 *   bytes  0-11  normal   (3 × float32)  — skipped for rendering
 *   bytes 12-23  vertex 0 (3 × float32)
 *   bytes 24-35  vertex 1 (3 × float32)
 *   bytes 36-47  vertex 2 (3 × float32)
 *   bytes 48-49  attr     (uint16)       — ignored
 *
 * out: (n_faces, 9) float32  [x0,y0,z0, x1,y1,z1, x2,y2,z2]
 */
extern "C" __global__
void read_binary_stl_faces(
        const unsigned char* __restrict__ data,
        long long n_faces,
        float*    __restrict__ out)
{
    long long tid = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= n_faces) return;

    const unsigned char* rec = data + tid * 50LL + 12LL; // skip normal
    float* dst = out + tid * 9LL;

    #pragma unroll
    for (int i = 0; i < 9; i++) {
        unsigned int bits = (unsigned int)rec[i*4]
                          | (unsigned int)rec[i*4+1] << 8
                          | (unsigned int)rec[i*4+2] << 16
                          | (unsigned int)rec[i*4+3] << 24;
        dst[i] = __uint_as_float(bits);
    }
}
"""

env: CUDA_VISIBLE_DEVICES=4   # set cuda device (optional)


In [3]:
def load_stl_gpu(path: Path) -> np.ndarray:
    """
    Load binary STL vertices using GPU.
    Returns float32 array of shape (N, 9):
        columns 0-2  vertex 0
        columns 3-5  vertex 1
        columns 6-8  vertex 2
    """
    import cupy as cp
    cp.cuda.set_allocator(None)   # bypass memory pool → real cudaFree on del

    file_size = path.stat().st_size
    with open(path, "rb") as f:
        f.seek(80)
        n_faces = struct.unpack("<I", f.read(4))[0]

    HEADER     = 84
    FACE_BYTES = 50
    data_bytes = n_faces * FACE_BYTES

    free_vram = cp.cuda.runtime.memGetInfo()[0]
    dev_name  = cp.cuda.runtime.getDeviceProperties(0)["name"].decode()
    print(f"GPU        : {dev_name}")
    print(f"Free VRAM  : {free_vram/1e9:.1f} GB")
    print(f"STL size   : {file_size/1e6:.0f} MB  |  {n_faces:,} faces")

    # Check VRAM: need data_bytes (input) + n_faces*36 (output float32 vertices)
    need = data_bytes + n_faces * 36
    if need > free_vram * 0.85:
        # Process in two halves
        print(f"[INFO] Large file — processing in 2 halves to fit VRAM")
        half = n_faces // 2
        a = _load_chunk_gpu(path, HEADER, half, 0, cp)
        b = _load_chunk_gpu(path, HEADER, n_faces - half, half, cp)
        verts = np.concatenate([a, b], axis=0)
    else:
        verts = _load_chunk_gpu(path, HEADER, n_faces, 0, cp)

    print(f"Loaded     : {len(verts):,} faces")
    return verts


def _load_chunk_gpu(path, header_offset, n, face_offset, cp):
    import cupy as cp

    FACE_BYTES = 50
    mod    = cp.RawModule(code=_CUDA_READ_FACES)
    kernel = mod.get_function("read_binary_stl_faces")
    BLOCK  = 256

    # Read chunk from disk
    byte_offset = header_offset + face_offset * FACE_BYTES
    byte_count  = n * FACE_BYTES
    with open(path, "rb") as f:
        f.seek(byte_offset)
        raw = f.read(byte_count)

    t0 = time.time()
    print(f"  uploading {byte_count/1e6:.0f} MB to GPU ...", end=" ", flush=True)
    gpu_data = cp.frombuffer(raw, dtype=cp.uint8).copy()
    del raw
    cp.cuda.runtime.deviceSynchronize()
    print(f"{time.time()-t0:.1f}s", flush=True)

    # Run kernel
    gpu_out = cp.empty(n * 9, dtype=cp.float32)
    grid    = (int(np.ceil(n / BLOCK)),)
    t0 = time.time()
    print(f"  parsing {n:,} faces on GPU ...", end=" ", flush=True)
    kernel(grid, (BLOCK,), (gpu_data, np.int64(n), gpu_out))
    cp.cuda.runtime.deviceSynchronize()
    print(f"{time.time()-t0:.1f}s", flush=True)
    del gpu_data

    # Download
    t0 = time.time()
    print(f"  downloading to CPU ...", end=" ", flush=True)
    verts = cp.asnumpy(gpu_out).reshape(n, 9)
    del gpu_out
    cp.cuda.runtime.deviceSynchronize()
    print(f"{time.time()-t0:.1f}s", flush=True)

    return verts




In [4]:
# ── [Cell 4] Decimate ─────────────────────────────────────────────────────────

def decimate(verts: np.ndarray, max_faces: int) -> np.ndarray:
    """Uniform subsampling — keeps every Nth face."""
    n = len(verts)
    if n <= max_faces:
        print(f"No decimation needed ({n:,} faces ≤ {max_faces:,})")
        return verts
    step = int(np.ceil(n / max_faces))
    out  = verts[::step]
    print(f"Decimated  : {n:,} → {len(out):,} faces  (kept 1/{step})")
    return out




In [5]:
# ── [Cell 5] Build k3d mesh arrays ───────────────────────────────────────────

def to_k3d_arrays(verts: np.ndarray):
    """
    Convert (N, 9) face array to k3d vertices + indices.
    k3d expects:
        vertices : float32 (M, 3)  unique or repeated vertex positions
        indices  : uint32  (N, 3)  triangle indices into vertices
    Fastest path: treat each face's 3 verts as independent (no dedup).
    """
    n = len(verts)
    # verts shape (N, 9) → (N*3, 3)
    positions = verts.reshape(-1, 3).astype(np.float32)
    indices   = np.arange(n * 3, dtype=np.uint32).reshape(n, 3)
    return positions, indices




In [6]:
# ── [Cell 6] Render ───────────────────────────────────────────────────────────

def render(positions: np.ndarray, indices: np.ndarray):
    """Render mesh in Jupyter using k3d."""
    try:
        import k3d
    except ImportError:
        sys.exit("k3d not found.  Install with:  pip install k3d")

    print(f"Rendering  : {len(indices):,} faces  |  "
          f"{len(positions):,} vertices", flush=True)

    # Compute bounding box for camera framing
    bbox_min = positions.min(axis=0)
    bbox_max = positions.max(axis=0)
    center   = ((bbox_min + bbox_max) / 2).tolist()
    extent   = float(np.linalg.norm(bbox_max - bbox_min))
    print(f"Bounds     : {bbox_min.tolist()}  →  {bbox_max.tolist()}")
    print(f"Extent     : {extent:.2f} mm")

    plot = k3d.plot(
        background_color=BACKGROUND,
        camera_auto_fit=True,
        grid_visible=False,
    )

    mesh = k3d.mesh(
        vertices  = positions,
        indices   = indices,
        color     = SURFACE_COLOR,
        opacity   = OPACITY,
        flat_shading = False,
        name      = STL_PATH.stem,
    )
    plot += mesh
    plot.display()
    return plot




In [ ]:
# ── [Cell 7] Run everything ───────────────────────────────────────────────────

if __name__ == "__main__" or "get_ipython" in dir():
    t_total = time.time()

    print("=" * 55)
    print("Step 1/4  Load binary STL via GPU")
    print("=" * 55)
    verts = load_stl_gpu(STL_PATH)

    print("\n" + "=" * 55)
    print("Step 2/4  Decimate for rendering")
    print("=" * 55)
    verts_dec = decimate(verts, MAX_FACES)
    del verts   # free ~large array

    print("\n" + "=" * 55)
    print("Step 3/4  Build mesh arrays")
    print("=" * 55)
    positions, indices = to_k3d_arrays(verts_dec)

    print("\n" + "=" * 55)
    print("Step 4/4  Render in Jupyter (k3d)")
    print("=" * 55)
    plot = render(positions, indices)

    print(f"\nTotal time : {time.time()-t_total:.1f}s")
    print("Done. Rotate the 3D widget above with mouse drag.")

Step 1/4  Load binary STL via GPU
GPU        : NVIDIA L40S
Free VRAM  : 47.2 GB
STL size   : 18376 MB  |  367,510,072 faces
  uploading 18376 MB to GPU ... 24.4s
  parsing 367,510,072 faces on GPU ... 0.0s
  downloading to CPU ... 1.9s
Loaded     : 367,510,072 faces

Step 2/4  Decimate for rendering
Decimated  : 367,510,072 → 1,997,338 faces  (kept 1/184)

Step 3/4  Build mesh arrays

Step 4/4  Render in Jupyter (k3d)
Rendering  : 1,997,338 faces  |  5,992,014 vertices
Bounds     : [0.0, 0.05866139382123947, 2.8502490520477295]  →  [34.388240814208984, 0.7048299908638, 3.4954981803894043]
Extent     : 34.40 mm
